# HAPS Geometry Control Panel

Change Gateway / relay-UAV placement and regenerate everything that depends on it
(figures, tables, the LSTM rain forecaster, and the DQN association agent) with
button clicks, instead of hand-editing geometry constants across six Python files
and re-running ~15 scripts in order.

**How it works:** Section 1 writes a small placement spec to `config/geometry.json`
via `src/models/geometry_config.py`, the single source of truth every other script
now reads from (instead of hardcoding its own copy of the Gateway/UAV geometry).
Sections 2-5 run the real pipeline scripts as subprocesses (using *this* kernel's
own Python interpreter) and show their output inline.

Run cells top to bottom once to build the UI; after that, every button is
independent -- click whichever ones you need, in any order (Section 1's "Apply
placement" first, if you're changing the geometry).


In [ ]:
import json
import os
import subprocess
import sys
import shutil
import glob
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, Image, HTML, clear_output

# Repo layout: this notebook lives in <repo_root>/notebooks/. Jupyter sets the kernel's cwd
# to the notebook's own directory, but detect the repo root by looking for src/models/ rather
# than assuming that, so this also works if the notebook is launched from the repo root.
_cwd = Path.cwd()
if (_cwd / "src" / "models").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "src" / "models").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise RuntimeError(
        f"Could not locate the repo root (src/models/) from cwd={_cwd}. "
        "Launch this notebook from the repo root or from notebooks/."
    )

SRC_DIR = REPO_ROOT / "src"
MODELS_DIR = SRC_DIR / "models"
GENERATORS_DIR = SRC_DIR / "generators"
LSTM_DIR = SRC_DIR / "lstm"
DQN_DIR = SRC_DIR / "drl_dqn"
FIGURES_DIR = REPO_ROOT / "figures"
APPENDIX_TABLES_DIR = REPO_ROOT / "appendix_tables"

# Use the same Python interpreter this notebook is running under -- NOT a bare "python" --
# so subprocess calls see the same installed packages (torch, numpy, itur, ...).
PY = sys.executable

if str(MODELS_DIR) not in sys.path:
    sys.path.insert(0, str(MODELS_DIR))
import geometry_config  # noqa: E402


def run_script(script_name, cwd, args=None, timeout=None):
    """Run `python <script_name> [args...]` with cwd=cwd, using this notebook's own
    interpreter. Always captures stdout/stderr (never swallowed) and returns the
    completed-process object."""
    cmd = [PY, script_name] + list(args or [])
    return subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True, timeout=timeout)


def print_result(label, result):
    """Print a subprocess result in full -- stdout, then stderr, then a pass/fail line.
    pdflatex-style silent failures aside, a nonzero return code is always surfaced here."""
    print(f"=== {label}  (exit code {result.returncode}) ===")
    if result.stdout:
        print("--- stdout ---")
        print(result.stdout)
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr)
    print(f"[{'OK' if result.returncode == 0 else 'FAILED'}] {label}")
    print()


print(f"Repo root:   {REPO_ROOT}")
print(f"Python exe:  {PY}")
print(f"geometry.json: {geometry_config.GEOMETRY_JSON_PATH} "
      f"({'exists' if os.path.exists(geometry_config.GEOMETRY_JSON_PATH) else 'will be created with defaults'})")


## 1. Placement controls

Choose how the Gateway and the relay UAV are placed, relative to the serving
HAPS's own nadir at (0, 0):

- **Centroid** -- fixed at the HAPS triangle's centroid (equidistant from all
  three HAPS). Today's Gateway default.
- **Near serving HAPS nadir -- good-sector azimuth** -- a small offset from the
  serving HAPS's own nadir, automatically along the "good sector" direction
  (away from the midpoint of the two interferer HAPS). Today's UAV default
  (`offset_km=0.1`).
- **Near serving HAPS nadir -- explicit azimuth** -- same, but at an azimuth
  (degrees) you specify directly.
- **Fully custom (x, y) km** -- any position, relative to the serving HAPS's
  nadir.

Click **Apply placement** to write `config/geometry.json` and immediately see
the resulting slant range / elevation angle / (Gateway) ITU-R P.838 rain
coefficients for both nodes -- sanity-check these *before* regenerating
anything downstream.


In [ ]:
MODE_OPTIONS = [
    ("Centroid (equidistant from all 3 HAPS)", "centroid"),
    ("Near serving HAPS nadir -- good-sector azimuth", "serving_nadir_good_sector"),
    ("Near serving HAPS nadir -- explicit azimuth", "serving_nadir_explicit"),
    ("Fully custom (x, y) km", "custom_xy"),
]


def ui_fields_from_spec(spec):
    """Reverse-map a geometry_config placement spec to (ui_mode, offset_km, azimuth_deg, x_km, y_km)."""
    mode = spec.get("mode", "centroid")
    if mode == "centroid":
        return "centroid", 0.1, 0.0, 0.0, 0.0
    if mode == "serving_nadir":
        offset_km = float(spec.get("offset_km", 0.1))
        if spec.get("azimuth_deg") is not None:
            return "serving_nadir_explicit", offset_km, float(spec["azimuth_deg"]), 0.0, 0.0
        return "serving_nadir_good_sector", offset_km, 0.0, 0.0, 0.0
    if mode == "custom_xy_km":
        return "custom_xy", 0.1, 0.0, float(spec.get("x_km", 0.0)), float(spec.get("y_km", 0.0))
    return "centroid", 0.1, 0.0, 0.0, 0.0


def spec_from_ui(fields):
    """Build a geometry_config placement spec from one node's widget values."""
    mode = fields["mode"].value
    if mode == "centroid":
        return {"mode": "centroid"}
    if mode == "serving_nadir_good_sector":
        return {"mode": "serving_nadir", "offset_km": float(fields["offset"].value), "azimuth_mode": "good_sector"}
    if mode == "serving_nadir_explicit":
        return {"mode": "serving_nadir", "offset_km": float(fields["offset"].value),
                "azimuth_deg": float(fields["azimuth"].value)}
    if mode == "custom_xy":
        return {"mode": "custom_xy_km", "x_km": float(fields["x"].value), "y_km": float(fields["y"].value)}
    raise ValueError(f"Unknown UI mode: {mode!r}")


def make_node_panel(node_label, spec):
    ui_mode, offset_km, azimuth_deg, x_km, y_km = ui_fields_from_spec(spec)

    mode_dd = widgets.Dropdown(options=MODE_OPTIONS, value=ui_mode, description=f"{node_label} mode:",
                                style={"description_width": "230px"}, layout=widgets.Layout(width="560px"))
    offset_box = widgets.FloatText(value=offset_km, description="offset_km:",
                                    layout=widgets.Layout(width="220px"))
    azimuth_box = widgets.FloatText(value=azimuth_deg, description="azimuth_deg:",
                                     layout=widgets.Layout(width="220px"))
    x_box = widgets.FloatText(value=x_km, description="x_km:", layout=widgets.Layout(width="220px"))
    y_box = widgets.FloatText(value=y_km, description="y_km:", layout=widgets.Layout(width="220px"))

    def update_visibility(*_args):
        mode = mode_dd.value
        offset_box.layout.display = "" if mode in ("serving_nadir_good_sector", "serving_nadir_explicit") else "none"
        azimuth_box.layout.display = "" if mode == "serving_nadir_explicit" else "none"
        x_box.layout.display = "" if mode == "custom_xy" else "none"
        y_box.layout.display = "" if mode == "custom_xy" else "none"

    mode_dd.observe(update_visibility, names="value")
    update_visibility()

    panel = widgets.VBox([mode_dd, widgets.HBox([offset_box, azimuth_box, x_box, y_box])])
    fields = {"mode": mode_dd, "offset": offset_box, "azimuth": azimuth_box, "x": x_box, "y": y_box}
    return panel, fields


_current_geometry = geometry_config.load_geometry()
gw_panel, gw_fields = make_node_panel("Gateway", _current_geometry["gateway"])
uav_panel, uav_fields = make_node_panel("UAV", _current_geometry["uav"])

apply_button = widgets.Button(description="Apply placement", button_style="primary",
                               layout=widgets.Layout(width="200px"))
placement_output = widgets.Output()


def describe_placement(node, placement):
    line = (f"[{node}] mode={placement['mode']}  "
            f"xy=({placement['x_km']:.4f}, {placement['y_km']:.4f}) km  "
            f"horiz_offset={placement['horiz_offset_km']:.4f} km  "
            f"slant={placement['slant_km']:.2f} km  "
            f"elevation={placement['elevation_deg']:.2f} deg")
    if node == "gateway":
        line += f"  K_RAIN={placement['k_rain']:.4f}  ALPHA_RAIN={placement['alpha_rain']:.4f}"
    return line


def on_apply_clicked(_):
    with placement_output:
        clear_output()
        try:
            geometry = {"gateway": spec_from_ui(gw_fields), "uav": spec_from_ui(uav_fields)}
            path = geometry_config.save_geometry(geometry)
            print(f"Wrote {path}\n{json.dumps(geometry, indent=2)}\n")
            placements = geometry_config.compute_all(geometry)
            for node in ("gateway", "uav"):
                print(describe_placement(node, placements[node]))
            print(
                "\nSanity check before regenerating: elevation should be in [0, 90] deg and "
                "slant range should be at least the HAPS-to-node altitude difference (20 km "
                "minus 0.05 km for the Gateway, 20 km minus 0.5 km for the UAV)."
            )
        except Exception as exc:  # surface the error in the output widget, not just the kernel
            print(f"[FAILED] {type(exc).__name__}: {exc}")


apply_button.on_click(on_apply_clicked)

display(HTML("<h3>Gateway placement</h3>"), gw_panel,
        HTML("<h3>UAV placement</h3>"), uav_panel,
        apply_button, placement_output)

# Show today's already-configured geometry right away (read-only, no write), so there's
# something to compare against before the first click of "Apply placement".
with placement_output:
    print(f"Currently configured in {geometry_config.GEOMETRY_JSON_PATH} (click 'Apply placement' "
          f"above to change it):\n")
    _current_placements = geometry_config.compute_all(_current_geometry)
    for _node in ("gateway", "uav"):
        print(describe_placement(_node, _current_placements[_node]))


## 2. Regenerate figures & tables

Runs the three generator scripts that depend on the placement above, then
copies their outputs to the project-root `figures/` and `appendix_tables/`
directories (mirroring what was previously done by hand).


In [ ]:
regen_button = widgets.Button(description="Regenerate figures & tables", button_style="success",
                               layout=widgets.Layout(width="240px"))
regen_output = widgets.Output()

# These three scripts all use relative paths (os.makedirs('appendix_data', ...), etc.), so
# they must be run with cwd=src/ -- matching how they're documented/used elsewhere in this repo.
_REGEN_SCRIPTS = [
    os.path.join("generators", "generate_relay_table.py"),
    os.path.join("generators", "generate_uav_relay_table.py"),
    os.path.join("generators", "generate_constellation_diagram.py"),
]

# (script-relative source path under src/, destination under the project root)
_REGEN_COPIES = [
    (SRC_DIR / "output" / "constellation_with_sectors.png", FIGURES_DIR / "constellation_with_sectors.png"),
    (SRC_DIR / "output" / "12_uav_relay_comparison.png", FIGURES_DIR / "12_uav_relay_comparison.png"),
    (SRC_DIR / "appendix_tables" / "UAV_relay_path_comparison.tex", APPENDIX_TABLES_DIR / "UAV_relay_path_comparison.tex"),
    (SRC_DIR / "appendix_tables" / "Relay_path_comparison.tex", APPENDIX_TABLES_DIR / "Relay_path_comparison.tex"),
]


def on_regen_clicked(_):
    with regen_output:
        clear_output()
        for script in _REGEN_SCRIPTS:
            try:
                result = run_script(script, cwd=SRC_DIR)
            except Exception as exc:
                print(f"[FAILED] could not launch {script}: {type(exc).__name__}: {exc}")
                return
            print_result(script, result)
            if result.returncode != 0:
                print(f"[STOPPED] {script} failed -- not copying outputs or continuing.")
                return

        print("Copying regenerated outputs into the project-root figures/ and appendix_tables/ ...")
        for src, dst in _REGEN_COPIES:
            if not src.exists():
                print(f"[WARN] expected output not found, skipping copy: {src}")
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(src, dst)
            print(f"[OK] {src.relative_to(REPO_ROOT)}  ->  {dst.relative_to(REPO_ROOT)}")

        constellation_png = FIGURES_DIR / "constellation_with_sectors.png"
        if constellation_png.exists():
            print("\nRegenerated constellation diagram:")
            display(Image(filename=str(constellation_png)))


regen_button.on_click(on_regen_clicked)
display(HTML(
    "Runs (in order, cwd=src/): generate_relay_table.py, generate_uav_relay_table.py, "
    "generate_constellation_diagram.py -- then copies "
    "constellation_with_sectors.png / 12_uav_relay_comparison.png into figures/ and "
    "UAV_relay_path_comparison.tex / Relay_path_comparison.tex into appendix_tables/."
))
display(regen_button, regen_output)


## 3. Retrain LSTM (rain-attenuation forecaster)

The LSTM's training **label** is the Gateway's feeder-hop rain attenuation,
computed from the Gateway placement configured in Section 1. If you changed
that placement, retrain before trusting the forecast.


In [ ]:
lstm_button = widgets.Button(description="Retrain LSTM (base model)", button_style="warning",
                              layout=widgets.Layout(width="220px"))
lstm_output = widgets.Output()


def on_lstm_clicked(_):
    with lstm_output:
        clear_output()
        print("Running train_lstm.py (cwd=src/lstm) ...")
        try:
            result = run_script("train_lstm.py", cwd=LSTM_DIR)
        except Exception as exc:
            print(f"[FAILED] could not launch train_lstm.py: {type(exc).__name__}: {exc}")
            return
        print_result("train_lstm.py", result)


lstm_button.on_click(on_lstm_clicked)

# These all import their rain-attenuation constants (ELEV_DEG, K_RAIN, ALPHA_RAIN, ...) from
# train_lstm.py, so running train_lstm.py first (always first in this sequence) is what wires
# them to whatever Gateway placement is currently configured -- no separate config step needed.
_FULL_LSTM_PIPELINE = [
    "train_lstm.py",
    "train_lstm_earlystop.py",
    "train_final_lstm.py",
    "train_forward_lstm.py",
    "export_lstm_forecasts.py",
    "ensemble_weighted.py",
    "verify_2026.py",
    "plot_forecast_windows.py",
]

full_lstm_button = widgets.Button(description="Run full LSTM pipeline (slower)", button_style="danger",
                                   layout=widgets.Layout(width="260px"))
full_lstm_output = widgets.Output()

_FORECAST_SRC = REPO_ROOT / "data" / "processed" / "figures" / "forecast_comparison_300h.png"
_FORECAST_DST = FIGURES_DIR / "lstm_forecast_2026.png"


def on_full_lstm_clicked(_):
    with full_lstm_output:
        clear_output()
        for script in _FULL_LSTM_PIPELINE:
            print(f"Running {script} (cwd=src/lstm) ...")
            try:
                result = run_script(script, cwd=LSTM_DIR)
            except Exception as exc:
                print(f"[FAILED] could not launch {script}: {type(exc).__name__}: {exc}")
                return
            print_result(script, result)
            if result.returncode != 0:
                print(f"[STOPPED] {script} failed -- not continuing the pipeline.")
                return

        if _FORECAST_SRC.exists():
            _FORECAST_DST.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(_FORECAST_SRC, _FORECAST_DST)
            print(f"[OK] {_FORECAST_SRC.relative_to(REPO_ROOT)}  ->  {_FORECAST_DST.relative_to(REPO_ROOT)}")
            print("\nRegenerated forecast plot:")
            display(Image(filename=str(_FORECAST_DST)))
        else:
            print(f"[WARN] expected forecast figure not found at {_FORECAST_SRC}")


full_lstm_button.on_click(on_full_lstm_clicked)

display(HTML(
    "<b>Note:</b> the LSTM's rain-attenuation label is derived from the Gateway placement "
    "configured in Section 1 (train_lstm.py's ELEV_DEG / K_RAIN / ALPHA_RAIN, via "
    "models/geometry_config.py). If you changed the Gateway placement there, retrain here "
    "before trusting the forecast -- the previously-trained model's label no longer matches."
))
display(lstm_button, lstm_output)
display(HTML("<hr><b>Full pipeline</b> (base model + early-stop + final + forward + export + "
             "ensemble + verify + plot -- several models, slower):"))
display(full_lstm_button, full_lstm_output)


## 4. DQN association agent

From fast (no training) to genuinely slow (multi-hour GPU runs). The two slow
buttons are disabled until you tick their confirmation checkbox.


In [ ]:
display(HTML(
    "<b>Note:</b> the DQN scripts below train and evaluate on whatever episodes/geometry is "
    "currently configured via the Placement controls in Section 1 (through "
    "haps_association_env.py's RELAY_FEEDER_DIST_KM / RELAY_FEEDER_ELEVATION_DEG / uav_xy_m, "
    "which read config/geometry.json). Apply your desired placement there first."
))

# --- Sanity check (instant): run_phase1_table.py, no training -------------------------------
sanity_button = widgets.Button(description="Sanity check (instant)", button_style="info",
                                layout=widgets.Layout(width="220px"))
sanity_output = widgets.Output()


def on_sanity_clicked(_):
    with sanity_output:
        clear_output()
        print("Running run_phase1_table.py (cwd=src/drl_dqn) ...")
        try:
            result = run_script("run_phase1_table.py", cwd=DQN_DIR)
        except Exception as exc:
            print(f"[FAILED] could not launch run_phase1_table.py: {type(exc).__name__}: {exc}")
            return
        print_result("run_phase1_table.py", result)


sanity_button.on_click(on_sanity_clicked)
display(sanity_button, sanity_output)

# --- Single-seed real-episode run (~5 min): train_dqn_real.py -------------------------------
real_button = widgets.Button(description="Single-seed real-episode run (~5 min)", button_style="warning",
                              layout=widgets.Layout(width="280px"))
real_output = widgets.Output()


def on_real_clicked(_):
    with real_output:
        clear_output()
        print("Running train_dqn_real.py (cwd=src/drl_dqn) -- this can take several minutes...")
        try:
            result = run_script("train_dqn_real.py", cwd=DQN_DIR, timeout=1800)
        except Exception as exc:
            print(f"[FAILED] could not launch train_dqn_real.py: {type(exc).__name__}: {exc}")
            return
        print_result("train_dqn_real.py", result)


real_button.on_click(on_real_clicked)
display(real_button, real_output)

# --- Full multi-seed confirmation (SLOW, ~5 hours): run_multiseed.py, gated by a checkbox ----
display(HTML("<hr><span style='color:#b30000;font-weight:bold'>WARNING: multi-hour GPU run.</span>"))

confirm_multiseed = widgets.Checkbox(value=False, description="I understand this takes ~5 hours")
multiseed_button = widgets.Button(description="Full multi-seed confirmation (SLOW, ~5 hours)",
                                   button_style="danger", disabled=True,
                                   layout=widgets.Layout(width="320px"))
multiseed_output = widgets.Output()


def on_confirm_multiseed_change(change):
    multiseed_button.disabled = not change["new"]


confirm_multiseed.observe(on_confirm_multiseed_change, names="value")


def on_multiseed_clicked(_):
    with multiseed_output:
        clear_output()
        print("Running run_multiseed.py (cwd=src/drl_dqn) -- this will take hours, do not close the kernel...")
        try:
            result = run_script("run_multiseed.py", cwd=DQN_DIR, timeout=None)
        except Exception as exc:
            print(f"[FAILED] could not launch run_multiseed.py: {type(exc).__name__}: {exc}")
            return
        print_result("run_multiseed.py", result)


multiseed_button.on_click(on_multiseed_clicked)
display(confirm_multiseed, multiseed_button, multiseed_output)

# --- Ablations (SLOW, ~15-20 hours): run_ablations.py, gated by a checkbox ------------------
display(HTML("<hr><span style='color:#b30000;font-weight:bold'>WARNING: very long multi-hour GPU run.</span>"))

confirm_ablations = widgets.Checkbox(value=False, description="I understand this takes ~15-20 hours")
ablations_button = widgets.Button(description="Ablations (SLOW, ~15-20 hours)",
                                   button_style="danger", disabled=True,
                                   layout=widgets.Layout(width="320px"))
ablations_output = widgets.Output()


def on_confirm_ablations_change(change):
    ablations_button.disabled = not change["new"]


confirm_ablations.observe(on_confirm_ablations_change, names="value")


def on_ablations_clicked(_):
    with ablations_output:
        clear_output()
        print("Running run_ablations.py (cwd=src/drl_dqn) -- this will take many hours, do not close the kernel...")
        try:
            result = run_script("run_ablations.py", cwd=DQN_DIR, timeout=None)
        except Exception as exc:
            print(f"[FAILED] could not launch run_ablations.py: {type(exc).__name__}: {exc}")
            return
        print_result("run_ablations.py", result)


ablations_button.on_click(on_ablations_clicked)
display(confirm_ablations, ablations_button, ablations_output)


## 5. Compile the paper PDF

Pick any `.tex` file in the repo root and compile it with `pdflatex`, run
twice (same convention as the `Makefile`), locally, in your own Jupyter
kernel. Pass/fail is read from the `.log` file's content (pdflatex often
exits 0 even when it hit a recoverable error), not from the process exit
code.


In [ ]:
def _list_tex_files():
    return sorted(p.name for p in REPO_ROOT.glob("*.tex"))


_tex_files = _list_tex_files()
_default_tex = "HAPS_AI_HW_ChannelModel.tex" if "HAPS_AI_HW_ChannelModel.tex" in _tex_files else (
    _tex_files[0] if _tex_files else None)

tex_dropdown = widgets.Dropdown(options=_tex_files, value=_default_tex, description="Tex file:",
                                 style={"description_width": "100px"}, layout=widgets.Layout(width="420px"))
refresh_tex_button = widgets.Button(description="Refresh list", layout=widgets.Layout(width="110px"))
compile_button = widgets.Button(description="Compile PDF", button_style="primary",
                                 layout=widgets.Layout(width="150px"))
open_button = widgets.Button(description="Open PDF", button_style="info", disabled=True,
                              layout=widgets.Layout(width="150px"))
compile_output = widgets.Output()

_last_pdf_path = {"path": None}


def on_refresh_tex_clicked(_):
    tex_dropdown.options = _list_tex_files()


refresh_tex_button.on_click(on_refresh_tex_clicked)


def on_compile_clicked(_):
    with compile_output:
        clear_output()
        tex_name = tex_dropdown.value
        if not tex_name:
            print("[FAILED] No .tex file selected (none found in the repo root).")
            return
        base = os.path.splitext(tex_name)[0]
        log_path = REPO_ROOT / f"{base}.log"
        pdf_path = REPO_ROOT / f"{base}.pdf"
        open_button.disabled = True
        _last_pdf_path["path"] = None

        print(f"Compiling {tex_name} with pdflatex, twice (matching the Makefile's convention "
              f"so references/citations resolve), cwd={REPO_ROOT} ...")
        for pass_num in (1, 2):
            try:
                result = subprocess.run(
                    ["pdflatex", "-interaction=nonstopmode", tex_name],
                    cwd=str(REPO_ROOT), capture_output=True, text=True,
                )
            except FileNotFoundError:
                print("[FAILED] pdflatex was not found on PATH. Install a TeX distribution "
                      "(e.g. MiKTeX or TeX Live) and make sure pdflatex.exe is on PATH, then retry.")
                return
            except Exception as exc:
                print(f"[FAILED] {type(exc).__name__}: {exc}")
                return
            print(f"--- pass {pass_num}: pdflatex exit code {result.returncode} ---")

        # pdflatex's -interaction=nonstopmode keeps going on most errors and often still exits
        # 0, so the exit code alone is not trustworthy -- check the .log for real failure
        # markers instead, as asked: "Fatal error" or any "!" error-marker lines.
        if not log_path.exists():
            print(f"[FAILED] No log file found at {log_path} -- pdflatex may not have run at all.")
            return

        log_text = log_path.read_text(errors="ignore")
        has_fatal = "Fatal error" in log_text
        bang_lines = [ln for ln in log_text.splitlines() if ln.startswith("!")]

        if has_fatal or bang_lines:
            reason = "a 'Fatal error'" if has_fatal else f"{len(bang_lines)} '!' error marker(s)"
            print(f"[FAILED] pdflatex log shows {reason}. Tail of {log_path.name}:")
            print("\n".join(log_text.splitlines()[-60:]))
            return

        print("[OK] No 'Fatal error' and no '!' error markers found in the log.")
        if pdf_path.exists():
            size_kb = pdf_path.stat().st_size / 1024.0
            mtime = pdf_path.stat().st_mtime
            print(f"[OK] PDF: {pdf_path}  ({size_kb:.1f} KB)")
            _last_pdf_path["path"] = str(pdf_path)
            open_button.disabled = False
        else:
            print(f"[FAILED] Log looked clean but no PDF was produced at {pdf_path}.")


def on_open_clicked(_):
    path = _last_pdf_path.get("path")
    with compile_output:
        if path and os.path.exists(path):
            if sys.platform == "win32":
                os.startfile(path)  # noqa: S606 -- intentional, user-initiated, local file
                print(f"Opened {path}")
            else:
                print(f"os.startfile is Windows-only; open manually: {path}")
        else:
            print("No successfully-compiled PDF to open yet -- click 'Compile PDF' first.")


compile_button.on_click(on_compile_clicked)
open_button.on_click(on_open_clicked)

display(HTML(
    "Lists the repo-root .tex files (glob *.tex, refreshable). Compiling runs pdflatex "
    "twice, locally, in <i>your</i> Jupyter kernel when you click the button -- this cell "
    "only wires the button, it does not compile anything on its own."
))
display(widgets.HBox([tex_dropdown, refresh_tex_button]))
display(widgets.HBox([compile_button, open_button]))
display(compile_output)
